# 06 - ROI model
Measured build values + labelled assumptions -> annual value, platform cost, payback. Writes evidence/roi_model.md.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from dbx import sql, ws, CATALOG
print('workspace:', ws().config.host, '| catalog:', CATALOG, '| user:', ws().current_user.me().user_name)

workspace: https://fevm-serverless-stable-oj7eie.cloud.databricks.com | catalog: serverless_stable_oj7eie_catalog | user: erica.yuen@databricks.com


In [2]:
import roi_model
m, a, out = roi_model.compute()
pd.Series(m, name='measured')

docs_processed                                               100
key_clause_f1                                           0.764356
micro_f1                                                0.827479
auto_renewal_recall                                     0.482759
pct_auto_accepted                                        0.86452
cost_per_1k_docs_usd                                      108.64
high_risk_flags                                               62
auto_renewing_contracts                                       17
auto_renewing_within_90_days                                   4
uncapped_liability_contracts                                  25
eval_mlflow_run_id              c3d5551824c7460e8f811c5691c0154e
auto_renewing_share                                         0.17
uncapped_share                                              0.25
Name: measured, dtype: object

In [3]:
pd.DataFrame([(k, *v) for k, v in roi_model.ASSUMPTIONS.items()], columns=['assumption','value','unit','source'])

,assumption,value,unit,source
0,contracts_in_portfolio,5000.0,contracts,illustrative (mid-size corporate legal team vendor portfolio)
1,manual_review_minutes,45.0,min / contract,illustrative (spec default; first-pass review of a commercial contract)
2,ai_assisted_review_minutes,8.0,min / contract,illustrative (spec default; reviewer verifies extracted terms)
3,legal_cost_per_hour,150.0,USD / hour,illustrative (spec default; fully loaded in-house cost)
4,unwanted_auto_renewal_share,0.1,share of auto-renewals,illustrative (spec default)
5,unwanted_renewal_annual_value,25000.0,USD / renewal,illustrative (spec default)
6,platform_run_cost_per_year,36000.0,USD / year,"illustrative (app + Lakebase CU_1 + serverless SQL, ~USD 3K/month)"


In [4]:
pd.Series(out, name='output').map(lambda v: round(v, 2) if isinstance(v, float) else v)

reviewer_hours_saved_per_year              3083.33
review_cost_saved_usd                    462500.00
auto_renewing_contracts_in_portfolio        850.00
unwanted_renewals_caught                     41.03
unwanted_renewals_avoided_usd           1025862.07
ai_processing_cost_usd                      543.20
platform_cost_usd                         36543.20
gross_value_usd                         1488362.07
net_value_usd                           1451818.87
roi_multiple                                 40.73
payback_months                                0.29
Name: output, dtype: float64

In [5]:
md_text = roi_model.to_markdown(m, a, out)
open('../evidence/roi_model.md', 'w').write(md_text)
print(md_text)

# ROI model - Northwind Legal Cloud (illustrative)

_Computed 2026-11-18 03:26 UTC by `scripts/roi_model.py` (notebook `notebooks/06_roi_model.ipynb`) from the live workspace, catalog `serverless_stable_oj7eie_catalog`._

## Measured in this build

| measured value                            | value    | source                                                        |
|:------------------------------------------|:---------|:--------------------------------------------------------------|
| Docs processed                            | 100      | gold_documents                                                |
| Extraction F1, key clauses                | 0.764    | ml.eval_results (MLflow run c3d5551824c7460e8f811c5691c0154e) |
| Extraction F1, all mapped clauses (micro) | 0.827    | ml.eval_results                                               |
| Auto-renewal detection recall             | 0.483    | ml.eval_results (CUAD Renewal Term proxy)                     |
| % clauses auto-accepted